# Zadanie 3: Model surogatowy ODE

W tym zadaniu stworzymy model surogatowy ODE dla zjawiska dyfuzji-reakcji, eliminując zmienną przestrzenną. Porównamy wyniki tego modelu z oryginalnym modelem czasowo-przestrzennym. Następnie, nauczymy sieć neuronową na danych z oryginalnego modelu i porównamy jej predykcje.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'zad2')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from solvers import run_simulation
import torch
import torch.nn as nn

## 1. Model oryginalny (PDE) i Model surogatowy (ODE)

Oryginalny model to równanie różniczkowe cząstkowe (PDE) dyfuzji-reakcji:
$$ \frac{\partial u}{\partial t} = D \frac{\partial^2 u}{\partial x^2} + (p + qu)(1-u) $$

Aby stworzyć model surogatowy, uśredniamy równanie po zmiennej przestrzennej $x \in [0, 1]$. Przy warunkach brzegowych Neumanna, człon dyfuzyjny $\int_0^1 D u_{xx} dx = D[u_x]_0^1 = 0$. Upraszczając $\mathbb{E}[R(u)] \approx R(\mathbb{E}[u])$, gdzie $U(t) = \mathbb{E}[u(x,t)] = \int_0^1 u(x,t) dx$, otrzymujemy równanie różniczkowe zwyczajne (ODE):
$$ \frac{dU}{dt} = (p + qU)(1-U) $$

In [ ]:
# Parametry symulacji
D_param = 0.1
p_param = 0.01
q_param = 1.0
tend = 0.5
nx = 100
dt = 1e-4

# Uruchomienie symulacji oryginalnego modelu PDE
x_pde, u_pde_final, _ = run_simulation(
    solver='crank-nicolson',
    nx=nx,
    dt=dt,
    tend=tend,
    D=D_param,
    p=p_param,
    q=q_param,
    init_kind='gaussian'
)

# Obliczenie średniej z modelu PDE w czasie
num_steps = int(tend / dt)
u_pde_mean_over_time = np.zeros(num_steps)
u_current = run_simulation(solver='crank-nicolson', nx=nx, dt=dt, tend=0, D=D_param, p=p_param, q=q_param, init_kind='gaussian')[1]
u_pde_mean_over_time[0] = np.mean(u_current)

# Re-use the solver's internal loop logic to get snapshots
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from solvers import _neumann_laplacian, reaction
dx = 1.0 / (nx - 1)
A = _neumann_laplacian(nx, dx)
M_l = sp.eye(nx) - 0.5 * dt * D_param * A
M_r = sp.eye(nx) + 0.5 * dt * D_param * A
for i in range(1, num_steps):
    Ru = reaction(u_current, p_param, q_param)
    rhs = M_r.dot(u_current) + dt * Ru
    u_current = spla.spsolve(M_l, rhs)
    u_pde_mean_over_time[i] = np.mean(u_current)

t_pde = np.linspace(0, tend, num_steps)

# Rozwiązanie modelu surogatowego ODE
def surrogate_ode(t, U):
    return (p_param + q_param * U) * (1 - U)

u0_pde_mean = np.mean(run_simulation(solver='crank-nicolson', nx=nx, dt=dt, tend=0, D=D_param, p=p_param, q=q_param, init_kind='gaussian')[1])
t_span_ode = [0, tend]
sol_ode = solve_ivp(surrogate_ode, t_span_ode, [u0_pde_mean], dense_output=True)

t_ode = np.linspace(0, tend, 100)
u_ode = sol_ode.sol(t_ode).T

## 3. Sieć neuronowa jako model surogatowy

Teraz nauczymy prostą sieć neuronową, aby przewidywała ewolucję średniej wartości $U(t)$ na podstawie danych z modelu PDE.

In [ ]:
<VSCode.Cell language="markdown">

</VSCode.Cell>
<VSCode.Cell language="python">

</VSCode.Cell>
<VSCode.Cell language="markdown">

</VSCode.Cell>
<VSCode.Cell language="python">

</VSCode.Cell>
<VSCode.Cell language="markdown">

</VSCode.Cell>
<VSCode.Cell language="python">
# Przygotowanie danych do uczenia
X_train = torch.tensor(t_pde, dtype=torch.float32).view(-1, 1)
y_train = torch.tensor(u_pde_mean_over_time, dtype=torch.float32).view(-1, 1)

# Definicja modelu sieci neuronowej
class SurrogateNN(nn.Module):
    def __init__(self):
        super(SurrogateNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model = SurrogateNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Proces uczenia
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}')

# Predykcje modelu
with torch.no_grad():
    u_nn_pred = model(torch.tensor(t_ode, dtype=torch.float32).view(-1, 1)).numpy()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Porównanie wszystkich modeli
</VSCode.Cell>
<VSCode.Cell language="python">
plt.figure(figsize=(8, 6))
plt.plot(t_pde, u_pde_mean_over_time, label='Średnia z modelu PDE (dane uczące)')
plt.plot(t_ode, u_ode, label='Model surogatowy ODE', linestyle='--')
plt.plot(t_ode, u_nn_pred, label='Model surogatowy NN', linestyle=':')
plt.title('Porównanie wszystkich modeli')
plt.xlabel('Czas (t)')
plt.ylabel('Średnia wartość U(t)')
plt.grid(True)
plt.legend()
plt.show()
</VSCode.Cell>